# Prepare ONS population data for MSOAs

In this notebook, the avaliable mid-2024 ONS population data estimates are joined with the Greater London MSOAs that have been compiled previously. The final output should be a population value for each MSOA which will then be used later to create our comprehensive London Bakery dataset.

In [1]:
import pandas as pd
import geopandas as gpd
from pathlib import Path

In [2]:
MSOA_PATH = Path("../data/spatial/processed/london_msoa_2021.gpkg")

POPULATION_PATH = Path("../data/spatial/raw/ons_msoa_population_data.xlsx")

OUTPUT_PATH = Path("../data/spatial/processed/london_msoa_population_2024.csv")

# Inspecting ONS population dataset workbooks

Since the ONS population data is in xlsx format, the correct workbook name with the mid-2024 data should be identified and used for the join with the 1002 loaded London MSOA rows. A preview is also loaded to get a familiarity with the dataset and the location and names of the columns.

In [3]:
london_msoa = gpd.read_file(MSOA_PATH)

ons_population_workbook = pd.ExcelFile(POPULATION_PATH)

print("Population workbook sheets:")
print(ons_population_workbook.sheet_names)

print(f"\nLondon MSOA rows: {len(london_msoa)}")

Population workbook sheets:
['Cover sheet', 'Contents', 'Notes', 'Related publications', 'Mid-2022 MSOA 2021', 'Mid-2023 MSOA 2021', 'Mid-2024 MSOA 2021']

London MSOA rows: 1002


In [4]:
population_preview = pd.read_excel(POPULATION_PATH, sheet_name="Mid-2024 MSOA 2021", usecols="A:L", nrows=12)

population_preview

,"Estimates by single year of age and sex for 2021 Middle layer Super Output Areas, mid-2024",Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10,Unnamed: 11
0,This worksheet contains one table.,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,To turn off freeze panes select the 'View' rib...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,LAD 2023 Code,LAD 2023 Name,MSOA 2021 Code,MSOA 2021 Name,Total,F0,F1,F2,F3,F4,F5,F6
3,E06000001,Hartlepool,E02002483,Hartlepool 001,10705,50,51,64,49,66,44,45
4,E06000001,Hartlepool,E02002484,Hartlepool 002,11122,66,71,53,77,76,74,56
5,E06000001,Hartlepool,E02002485,Hartlepool 003,8129,40,37,38,39,53,51,52
6,E06000001,Hartlepool,E02002489,Hartlepool 007,7752,28,28,40,39,36,45,33
7,E06000001,Hartlepool,E02002490,Hartlepool 008,6137,33,38,42,52,40,47,34
8,E06000001,Hartlepool,E02002491,Hartlepool 009,6885,22,25,37,19,33,47,37
9,E06000001,Hartlepool,E02002492,Hartlepool 010,6965,27,28,33,28,38,58,38


# Load required mid-2024 ONS population data

Since the column names for the tables begin on the fourth Excel row, it is set to header=3. Only the MSOA code and the total population of the MSOA code needs to be loaded.

In [5]:
ons_population = pd.read_excel(
    POPULATION_PATH,
    sheet_name="Mid-2024 MSOA 2021",
    header=3,
    usecols=["MSOA 2021 Code", "Total"],
    dtype={"MSOA 2021 Code": "string"}
)

ons_population = ons_population.rename(
    columns={"MSOA 2021 Code": "msoa_code",
             "Total": "population"}
    )

print("Population rows:", len(ons_population))
print("Missing MSOA codes:", ons_population["msoa_code"].isna().sum())
print("Duplicate MSOA codes:", ons_population["msoa_code"].duplicated().sum())

Population rows: 7264
Missing MSOA codes: 0
Duplicate MSOA codes: 0


# Limit population data to only that of Greater London

Filtering the ONS population dataset to only MSOA codes found in Greater London leaves one population record for each of the 1002 MSOAs.

In [6]:
london_msoa_population = (
    ons_population
    .loc[ons_population["msoa_code"].isin(london_msoa["msoa_code"])]
    .reset_index(drop=True))

print("London population rows:", len(london_msoa_population))
london_msoa_population.head()

London population rows: 1002


,msoa_code,population
0,E02000001,15111
1,E02000002,8820
2,E02000003,12655
3,E02000004,7056
4,E02000005,11630


# Validating Greater London population table

A final check is done on the table to confirm all of the correct 1002 London MSOAs are present and population values are logical.

In [7]:
missing_london_codes = set(london_msoa["msoa_code"]) - set(london_msoa_population["msoa_code"])

extra_codes = set(london_msoa_population["msoa_code"]) - set(london_msoa["msoa_code"])

print(f"London MSOA rows: {len(london_msoa)}")
print(f"London population by MSOA rows: {len(london_msoa_population)}")
print(f"Duplicate MSOA codes: {london_msoa_population["msoa_code"].duplicated().sum()}")
print(f"Non-London MSOAs in output: {len(extra_codes)}")

print(f"\nMissing population counts: {london_msoa_population["population"].isna().sum()}")
print(f"London MSOAs missing population counts: {len(missing_london_codes)}")
print(f"Non-positive population counts: {(london_msoa_population["population"] <= 0).sum()}")

print(f"\nTotal London population: {london_msoa_population["population"].sum()}")

London MSOA rows: 1002
London population by MSOA rows: 1002
Duplicate MSOA codes: 0
Non-London MSOAs in output: 0

Missing population counts: 0
London MSOAs missing population counts: 0
Non-positive population counts: 0

Total London population: 9089736


# Save validated Greater London population table

In [8]:
london_msoa_population.to_csv(OUTPUT_PATH, index=False)

print(f"Saved to: {OUTPUT_PATH}")

Saved to: ..\data\spatial\processed\london_msoa_population_2024.csv
